# Human-in-the-Loop: Mga Pre-Action Gate, Risk Tiering, at Audit Logging

Ang README para sa araling ito ay nagpapakilala ng Human-in-the-Loop gamit ang isang maikling snippet na nagtatanong sa user na `APPROVE` o `REJECT` matapos makabuo ng sagot ang ahente. Ang pattern na iyon ay isang magandang panimulang punto, ngunit ang mga production HITL implementations ay karaniwang nangangailangan ng tatlong karagdagang bahagi:

1. Isang **pre-action gate** na tumatakbo **bago** isagawa ng ahente ang isang mapanganib na hakbang, upang manatiling kontrolado ang gastos, hindi maibabalik na aksyon, at pagkaantala.
2. **Risk tiering**, upang ang mga mababang-risk na aksyon ay awtomatikong isagawa, ang mga medium-risk na aksyon ay ma-batch na aprubahan, at tanging ang mga high-risk na aksyon lamang ang humihinto para sa isang tao.
3. Isang **audit log kasama ang revision loop**, upang bawat desisyon ng gate ay naitatala bilang JSONL, at ang pagtanggi ay muling nagtatanong sa ahente gamit ang isang istrukturadong dahilan sa halip na basta mag-print lang ng `Revising...`.

Itinayo ng notebook na ito ang bawat isa sa mga ito gamit ang parehong mga primitives tulad ng sa `06-system-message-framework.ipynb`. Tumakbo ito nang end-to-end sa `DEMO_MODE = True` (hindi kailangan ng interactive na input) o gamit ang totoong mga prompt na `input()` kapag `DEMO_MODE = False`. Tandaan: sa DEMO_MODE ang retry ng ikatlong layunin ay scripted kaya ang mekaniks ng loop ay makikita nang buo. Ang totoong revision-driven na re-classification ay nangangailangan ng `DEMO_MODE = False` at isang operator.

**Hindi sakop (hinihawakan sa ibang mga aralin):** authentication at access control (Lesson 06 README threat 2), tool-call middleware (Lesson 14 MAF deep dive), mga pattern ng multi-agent debate.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Pattern 1: Pre-aksyon na gate

Ang snippet ng HITL sa README ay tinatawag muna ang ahente, pagkatapos ay hinihiling sa gumagamit na aprubahan ang output. Iyan ay isang **post-aksyon** na daloy. Ang ahente ay naipatupad na, kaya ang gastos sa pagtawag sa LLM ay nabayaran na, at anumang side effect (naipadalang email, naisulat na hilera sa database, naipost na komento) ay nangyari na.

Ang isang **pre-aksyon** na daloy ay naglalagay ng gate bago tumakbo ang ahente sa mapanganib na hakbang. Iminumungkahi ng ahente ang aksyon, ang gate ang nagpasiya kung ito ay isasagawa, at tanging sa pag-apruba lang nagaganap ang side effect.

| Aspeto | Post-aksyon na aprubal (snippet ng README) | Pre-aksyon na gate (notebook na ito) |
|---|---|---|
| Kailan tumatakbo ang aprubal? | Pagkatapos makagawa ng output ang ahente | Bago maganap ang anumang side-effect |
| Gastos sa LLM sa pagtanggi | Nababayaran na | Binabayaran lamang para sa mungkahi, hindi sa aksyon |
| Hindi mababaling mga side effect | Posible (nagawa na ang aksyon) | Pinipigilan |
| Kalinawan sa audit | Ang aprubal ay isang print statement | Ang aprubal ay isang JSON record na may timestamp, aksyon, dahilan |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: Risk tiering

Not every action needs human approval. A read-only lookup against a public API has different stakes than sending a customer email. Treating both the same wastes operator attention and slows the agent.

A simple 3-tier model:

| Tier | Examples | Approval flow |
|---|---|---|
| `low` (read-only) | Search a knowledge base, look up flight options, fetch a public web page | Auto-execute, logged for audit |
| `medium` (cheap mutation) | Cache a result, increment a counter, schedule a reminder | Auto-execute, but batched daily review |
| `high` (external-facing or irreversible) | Send an email, charge a card, post to a public channel | Block on human approval |

This is one tiering. Production systems often use more granular tiers (e.g., AWS IAM permission levels, role-based access tiers). The 3-tier version below is the smallest useful version for an agent that mixes read-only and side-effecting actions.

The classifier below uses keyword heuristics so the demo stays deterministic and cheap. In a production system you would swap this for a learned classifier or a policy engine.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Pattern 3: Audit log at revision loop

Ang `print("Response approved.")` ay hindi isang audit log. Para sa tiwala, ang bawat desisyon ng gate ay dapat naitala bilang isang istrukturadong pangyayari na maaari mong i-query, i-replay, o i-attach sa isang pagsusuri ng insidente sa hinaharap.

Dalawang bahagi:

1. **Append-only JSONL.** Isang linya kada desisyon, kasama ang timestamp, aksyon, tier, desisyon, dahilan. Madaling i-grep, madaling ipadala sa isang totoong log store sa hinaharap.
2. **Revision loop sa pagtanggi.** Kapag nagbalik ang gate ng `deny`, muling hinihikayat ng ahente ang sarili nitong mag-prompt gamit ang dahilan ng pagtanggi sa konteksto, upang ang susunod na panukala ay maiwasan ang problema.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Karagdagang mga mapagkukunan

Ilang iba pang pampublikong proyekto ang nagpapatupad ng mga baryasyon ng mga pattern na HITL na ito. Ihambing ang mga pamamaraan upang malaman kung ano ang akma sa iyong stack:

- **LangChain** human-in-the-loop tool wrappers ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): mga drop-in tool wrapper na humihinto sa pagpapatupad para sa input ng tao.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); muling inayos ito sa AutoGen v0.4+): gumagamit ng papel ng ahente na espesipikong kumakatawan sa tao sa mga multi-agent na pag-uusap.
- **Microsoft Agent Framework (MAF)** function-invocation middleware ([docs](https://learn.microsoft.com/agent-framework/)): middleware na tumatakbo sa paligid ng bawat tawag sa tool/fungsyon, na angkop para sa gating logic at mga daloy ng pag-apruba.

Ipinangangasiwa ng bawat proyekto nang iba ang tatlong sub-patterns: niwrap ng LangChain bilang mga tools, gumagamit ang AutoGen ng papel ng ahente, at gumagamit ang Microsoft Agent Framework ng function-invocation middleware. Basahin ang isa o dalawang implementasyon nang buong-buo bago pumili ng disenyo para sa sarili mong ahente.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
